# Race conditions
Protect shared state when concurrent tasks can update it.


In [ ]:
import asyncio

counter = 0
lock = asyncio.Lock()

async def increment() -> None:
    global counter
    async with lock:
        counter += 1

await asyncio.gather(*(increment() for _ in range(100)))
print(counter)


## Polished version
Place the lock beside the state and keep the critical section small.


In [ ]:
class OutOfStock(Exception):
    pass

class InventoryService:
    def __init__(self, stock: int) -> None:
        self.stock = stock
        self._lock = asyncio.Lock()

    async def reserve(self, quantity: int = 1) -> int:
        async with self._lock:
            if self.stock < quantity:
                raise OutOfStock
            self.stock -= quantity
            return self.stock

inventory = InventoryService(stock=2)
results = await asyncio.gather(inventory.reserve(), inventory.reserve())
print(results, inventory.stock)
